# Lab Assignment 5: Web Scraping
## DS 6001: Practice and Application of Data Science
### Robert Clay Harris

### Instructions
Please answer the following questions as completely as possible using text, code, and the results of code as needed. Format your answers in a Jupyter notebook. To receive full credit, make sure you address every part of the problem, and make sure your document is formatted in a clean and professional way.

For the following problems, you will be scraping http://books.toscrape.com/. This website is a fake book retailer, designed to mimic the design of many retail websites. It exists solely to help students practice web-scraping, so there aren’t going to be any ethical concerns with this particular exercise, and there shouldn’t be any issues with rate limits or other gates that could prevent web-scraping. Take a moment and look at this website, so that you know what you will be working with.

Your goal is to generate a dataframe with four columns: one for the title, one for the price, one for the star-rating, and one or the book cover JPEG’s URL. The dataframe will also 1000 rows, one for each of the 1000 books listed on the 50 pages of this website.

## Problem 0
Import the following libraries:

In [17]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import sys
sys.tracebacklimit = 0 # turn off the error tracebacks

## Problem 1
Pull the HTML code from http://books.toscrape.com/. Make sure you provide a user agent string. Then parse this HTML code and save the parsed code as a separate Python variable. [3 points]

In [18]:
r = requests.get('https://httpbin.org/user-agent')
useragent = r.json()['user-agent']

headers = {
    'User-Agent': useragent,
    'From': 'jbm2rt@virginia.edu'
}

response = requests.get('http://books.toscrape.com/', headers=headers)
html_code = response.text

# Parse the HTML code and save
parsed_html = BeautifulSoup(html_code, "html.parser")

### Problem 2
Extract all 20 of the book titles and save them in a list. [2 points]

In [19]:
# Extract titles
book_titles = [
    book.find('h3').find('a')['title']
    for book in parsed_html.find_all('article', class_='product_pod')
]

print(book_titles)

['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind', 'The Requiem Red', 'The Dirty Little Secrets of Getting Your Dream Job', 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 'The Black Maria', 'Starving Hearts (Triangular Trade Trilogy, #1)', "Shakespeare's Sonnets", 'Set Me Free', "Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 'Rip it Up and Start Again', 'Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991', 'Olio', 'Mesaerion: The Best Science Fiction Stories 1800-1849', 'Libertarianism for Beginners', "It's Only the Himalayas"]


### Problem 3
Extract the price of each of the 20 books and save these prices in a list. (The prices are listed in British pounds, and include the £ symbol. Remove the £ symbols: if you’ve saved the prices in a list named `prices`, then the following code should work: `prices = [s.replace('Â£', '') for s in prices]`.) [2 points]

In [20]:
# Extract the prices
prices = [
    book.find('p', class_='price_color').text
    for book in parsed_html.find_all('article', class_='product_pod')
]

# Remove the pound symbol
prices = [s.replace('Â£', '') for s in prices]

print(prices)

['51.77', '53.74', '50.10', '47.82', '54.23', '22.65', '33.34', '17.93', '22.60', '52.15', '13.99', '20.66', '17.46', '52.29', '35.02', '57.25', '23.88', '37.59', '51.33', '45.17']


## Problem 4
Extract the star level ratings for the 20 books. [Hint: for tags such as `<p class="star-rating One">` in which the class has a space, the class is actually a list in which the first item in the list is `"star-rating"` and the second item in the list is `"One"`. It's possible to search on either item in this list.] [3 points]

In [21]:
# Extract star level
star_ratings = [
    book.find('p', class_='star-rating')['class'][1]
    for book in parsed_html.find_all('article', class_='product_pod')
]

print(star_ratings)

['Three', 'One', 'One', 'Four', 'Five', 'One', 'Four', 'Three', 'Four', 'One', 'Two', 'Four', 'Five', 'Five', 'Five', 'Three', 'One', 'One', 'Two', 'Two']


## Problem 5
Extract the URLs for the JPEG thumbnail images that show the covers of the 20 books. (Maybe we want to mine the images to build models that predict the star level, literally judging books by their covers.) [2 points]

In [22]:
# Base url
base_url = 'http://books.toscrape.com/'

thumbnail_urls = [
    base_url + book.find('img')['src'].replace('../../', '')
    for book in parsed_html.find_all('article', class_='product_pod')
]

print(thumbnail_urls)

['http://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg', 'http://books.toscrape.com/media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg', 'http://books.toscrape.com/media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg', 'http://books.toscrape.com/media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg', 'http://books.toscrape.com/media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg', 'http://books.toscrape.com/media/cache/68/33/68339b4c9bc034267e1da611ab3b34f8.jpg', 'http://books.toscrape.com/media/cache/92/27/92274a95b7c251fea59a2b8a78275ab4.jpg', 'http://books.toscrape.com/media/cache/3d/54/3d54940e57e662c4dd1f3ff00c78cc64.jpg', 'http://books.toscrape.com/media/cache/66/88/66883b91f6804b2323c8369331cb7dd1.jpg', 'http://books.toscrape.com/media/cache/58/46/5846057e28022268153beff6d352b06c.jpg', 'http://books.toscrape.com/media/cache/be/f4/bef44da28c98f905a3ebec0b87be8530.jpg', 'http://books.toscrape.com/media/cache/10/48/1048f63d3b5061cd2f424d20b3f9b6

## Problem 6
Create a dataframe with one row for each of the 20 books, and the book titles, prices, star ratings, and cover JPEG URLs as the four columns. [2 points]

In [23]:
# Create dictionary
data = {
    'Book Title': book_titles,
    'Price': prices,
    'Star Rating': star_ratings,
    'Cover URL': thumbnail_urls
}

# Create DataFrame
df = pd.DataFrame(data)

print(df)

                                           Book Title  Price Star Rating  \
0                                A Light in the Attic  51.77       Three   
1                                  Tipping the Velvet  53.74         One   
2                                          Soumission  50.10         One   
3                                       Sharp Objects  47.82        Four   
4               Sapiens: A Brief History of Humankind  54.23        Five   
5                                     The Requiem Red  22.65         One   
6   The Dirty Little Secrets of Getting Your Dream...  33.34        Four   
7   The Coming Woman: A Novel Based on the Life of...  17.93       Three   
8   The Boys in the Boat: Nine Americans and Their...  22.60        Four   
9                                     The Black Maria  52.15         One   
10     Starving Hearts (Triangular Trade Trilogy, #1)  13.99         Two   
11                              Shakespeare's Sonnets  20.66        Four   
12          

## Problem 7
Create a function that takes the URL of the webpage to scrape as an input, applies the code you wrote for questions 1 through 6, and generates the dataframe from question 6 as the output. [3 points]

In [24]:
def scrape_books(url):
    r = requests.get('https://httpbin.org/user-agent')
    useragent = r.json()['user-agent']
    
    # Define headers
    headers = {
        'User-Agent': useragent,
        'From': 'jbm2rt@virginia.edu'
    }
    
    # Pull the HTML code
    response = requests.get(url, headers=headers)
    html_code = response.text
    
    # Parse the HTML
    parsed_html = BeautifulSoup(html_code, "html.parser")
    
    # Extract the 20 books
    books = parsed_html.find_all('article', class_='product_pod')
    
    # Extract title
    book_titles = [
        book.find('h3').find('a')['title']
        for book in books
    ]
    
    # Extract prices
    prices = [
        book.find('p', class_='price_color').text
        for book in books
    ]
    prices = [s.replace('Â£', '') for s in prices]
    
    # Extract star ratings
    star_ratings = [
        book.find('p', class_='star-rating')['class'][1]
        for book in books
    ]
    
    # Extract the cover JPEG URLs
    base_url = 'http://books.toscrape.com/'
    thumbnail_urls = [
        base_url + book.find('img')['src'].replace('../../', '')
        for book in books
    ]
    
    # Create a DataFrame
    data = {
        'Book Title': book_titles,
        'Price': prices,
        'Star Rating': star_ratings,
        'Cover URL': thumbnail_urls
    }
    df = pd.DataFrame(data)
    return df

## Problem 8
Notice that there are many pages to http://books.toscrape.com/. When you click on “Next” in the bottom-right corner of the screen, it takes you to http://books.toscrape.com/catalogue/page-2.html. The front page is the same as http://books.toscrape.com/catalogue/page-1.html, and there are 50 total pages.

Write a loop that uses the function you wrote in question 7 to scrape each of the 50 pages, and append each of these data frames together. If you write this loop correctly, your dataframe will have 1000 rows (20 books on each of the 50 pages). 

Some hints:

* Typing `new_df = pd.DataFrame()` with nothing in the parentheses will create an empty data frame on which new data can be appended.

* There are many loops you can use, but the most straightforward one is a for-values loop that counts from 1 to 50. In Python, you can initialize such a loop with for i in range(1, 51):, and indenting every line below it that belongs inside the loop. Inside the loop, the letter i is now a stand-in for the number currently being considered.

* You will need to figure out how to replace the number in URLs like http://books.toscrape.com/catalogue/page-2.html with the number currently under consideration in the loop. You might need the `str()` function, which turns numeric values into strings.

[3 points]


In [25]:
# Initialize empty DataFrame
new_df = pd.DataFrame()

# Loop through pages 1 to 50
for i in range(1, 51):
    # Construct URL
    url = "http://books.toscrape.com/catalogue/page-" + str(i) + ".html"
    
    # Scrape using function
    df_page = scrape_books(url)
    
    # Concat to new_df
    new_df = pd.concat([new_df, df_page], ignore_index=True)

In [26]:
print(new_df.head(3))
print("Total rows:", len(new_df))

             Book Title  Price Star Rating  \
0  A Light in the Attic  51.77       Three   
1    Tipping the Velvet  53.74         One   
2            Soumission  50.10         One   

                                           Cover URL  
0  http://books.toscrape.com/../media/cache/2c/da...  
1  http://books.toscrape.com/../media/cache/26/0c...  
2  http://books.toscrape.com/../media/cache/3e/ef...  
Total rows: 1000
